In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from collections import defaultdict
import json
import random
import os
import requests

from PIL import Image, ImageDraw

In [2]:
# Read the JSON file
def read_anime_recommendations(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Extract anime information in a more usable format
def extract_anime_info(data):
    all_anime = []
    
    # Iterate through each key-value pair in the dictionary
    for anime_id, recommendations in data.items():
        # Iterate through each recommendation in the list
        for rec in recommendations:
            anime_info = {
                'source_id': anime_id,  # The key in the outer dictionary
                'anime_id': rec['node']['id'],
                'title': rec['node']['title'],
                'image_medium': rec['node']['main_picture']['medium'],
                'image_large': rec['node']['main_picture']['large'],
                'num_recommendations': rec['num_recommendations']
            }
            all_anime.append(anime_info)
    
    return all_anime

# Convert to DataFrame if needed
def convert_to_dataframe(anime_list):
    import pandas as pd
    return pd.DataFrame(anime_list)

In [3]:
anime = pd.read_csv('backupdata/anime_data.csv')
anime_recs = read_anime_recommendations('anime_recommendations_full.json')

In [7]:
import os
import requests
import json

def download_anime_images(anime_df, folder="images", size="medium"):
    """
    Download anime thumbnails from 'main_picture' URLs.
    """
    os.makedirs(folder, exist_ok=True)

    # Fix: parse main_picture if needed
    if isinstance(anime_df.iloc[0]['main_picture'], str):
        anime_df['main_picture'] = anime_df['main_picture'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )

    for idx, row in anime_df.iterrows():
        anime_id = row['id']
        pic_info = row.get('main_picture')

        if isinstance(pic_info, dict) and pic_info.get(size):
            url = pic_info[size]
            filename = os.path.join(folder, f"{anime_id}.jpg")
            
            # Skip if already downloaded
            if os.path.exists(filename):
                continue

            print(f"Attempting to download for anime_id {anime_id}")
            
            try:
                response = requests.get(url, timeout=10)
                response.raise_for_status()
                with open(filename, "wb") as f:
                    f.write(response.content)
                print(f"Downloaded {anime_id}")
            except Exception as e:
                print(f"Failed {anime_id}: {e}")


In [8]:
download_anime_images(anime)

Attempting to download for anime_id 52991
Downloaded 52991
Attempting to download for anime_id 5114
Downloaded 5114
Attempting to download for anime_id 9253
Downloaded 9253
Attempting to download for anime_id 38524
Downloaded 38524
Attempting to download for anime_id 60022
Downloaded 60022
Attempting to download for anime_id 28977
Downloaded 28977
Attempting to download for anime_id 39486
Downloaded 39486
Attempting to download for anime_id 11061
Downloaded 11061
Attempting to download for anime_id 9969
Downloaded 9969
Attempting to download for anime_id 15417
Downloaded 15417
Attempting to download for anime_id 820
Downloaded 820
Attempting to download for anime_id 41467
Downloaded 41467
Attempting to download for anime_id 34096
Downloaded 34096
Attempting to download for anime_id 43608
Downloaded 43608
Attempting to download for anime_id 42938
Downloaded 42938
Attempting to download for anime_id 4181
Downloaded 4181
Attempting to download for anime_id 918
Downloaded 918
Attempting to

In [16]:
def crop_circle(input_path, output_path):
    img = Image.open(input_path).convert("RGBA")
    size = img.size

    mask = Image.new('L', size, 0)
    draw = ImageDraw.Draw(mask) 
    draw.ellipse((0, 0) + size, fill=255)

    result = Image.new('RGBA', size)
    result.paste(img, mask=mask)

    # Force output to be PNG
    output_path = os.path.splitext(output_path)[0] + ".png"
    result.save(output_path)


In [17]:
# go through the images folder and save the cropped images in a new folder
def crop_all_images(input_folder="images", output_folder="cropped_images"):
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".jpg"):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            crop_circle(input_path, output_path)

In [18]:
crop_all_images()